In [ ]:
import os
import google.generativeai as genai
from tavily import TavilyClient
from dotenv import load_dotenv

load_dotenv()

GOOGLE_API_KEY = os.getenv('GEMINI_API_KEY')
TAVILY_API_KEY = os.getenv('TAVILY_API_KEY')

d:\Proyectos\alianza-f1-knowledge-agent\Cursos\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\tcpad\AppData\Local\Temp\ipykernel_18256\1110624662.py:2: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


'Un sistema multiagente de Inteligencia Artificial consiste en múltiples agentes autónomos que colaboran para resolver problemas complejos. Cada agente tiene habilidades específicas y trabaja juntos para lograr objetivos comunes. Estos sistemas son útiles para tareas a gran escala y dinámicas.'

In [2]:
client = TavilyClient(api_key=TAVILY_API_KEY)
result = client.search("Que son multiagentes de Inteligencia Artificial?", 
                       include_answer=True)

result["answer"]

'Un sistema multiagente de Inteligencia Artificial consiste en múltiples agentes autónomos que colaboran para resolver problemas complejos. Cada agente tiene habilidades específicas y trabaja juntos para lograr objetivos comunes. Estos sistemas son útiles para tareas a gran escala y dinámicas.'

In [10]:
ciudad = "Italia"  # Puedes cambiar la ciudad según tus necesidades

query = f"""
Enumere los 5 principales restaurantes en {ciudad}, según evaluaciones recientes en poresto o sitios similares de turismo.
Para cada restaurante, indique:
- Tipo de cocina (ej.: regional, italiana, japonesa)
- Una breve descripción (máx. 2 líneas)
- Calificación promedio (si está disponible)
- Rango de precios

Responda únicamente con datos actualizados y relevantes para turistas.
"""

from ddgs import DDGS
import re

ddg = DDGS()

def search(query, max_results=6):
    try:
        results = ddg.text(query, max_results=max_results)
        return [i["href"] for i in results]
    except Exception as e:
        raise e

for link in search(query):
    print(link)

https://www.conociendoitalia.com/8-tipos-de-locales-donde-comer-en-italia/
https://blog-estudios.mundojoven.com/blog/los-mejores-restaurantes-en-italia/
https://es.hoteles.com/go/italia/ciudades-mas-gastronomicas-italia
https://www.20minutos.es/gastronomia/restaurantes/de-la-osteria-a-la-rosticceria-los-8-tipos-de-restaurantes-que-puedes-encontrar-en-italia-5118513/
https://maximavelocidad.com.ar/italian-restaurant-glen-alpine/
https://saltaconmigo.com/blog/2014/08/comer-en-italia-restaurantes-menu/


In [11]:
import requests
from bs4 import BeautifulSoup
from ddgs import DDGS
import re

ddg = DDGS()

def scrape_restaurantes_info(url):
    if not url:
        print("Error: URL vacia o no localizada.")
        return None

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, como Gecko) Chrome/138.0.0.0 Safari/537.36',
        'Accept-Language': 'es-MX,es;q=0.9,en;q=0.8'
    }

    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"Error al cargar la página {url}: {e}")
        return None
    
    soup = BeautifulSoup(response.text, 'html.parser')
    return soup

search_results = search(query)

if search_results:
    url = search_results[0]
else:
    url = None

soup = scrape_restaurantes_info(url)

print(f"Website: {url}\n\n")

if soup:
    print(str(soup.body.prettify())[:50000])
else:
    print("No fue posible raspar el contenido de la página o la URL no fue encontrada.")

Website: https://www.conociendoitalia.com/8-tipos-de-locales-donde-comer-en-italia/


<body class="wp-singular post-template-default single single-post postid-4002 single-format-standard wp-custom-logo wp-embed-responsive wp-theme-genesis wp-child-theme-magazine-pro header-image header-full-width content-sidebar genesis-breadcrumbs-hidden genesis-singular-image-hidden genesis-footer-widgets-hidden primary-nav" itemscope="" itemtype="https://schema.org/WebPage">
 <div class="site-container">
  <ul class="genesis-skip-link">
   <li>
    <a class="screen-reader-shortcut" href="#genesis-content">
     Saltar al contenido principal
    </a>
   </li>
   <li>
    <a class="screen-reader-shortcut" href="#genesis-nav-secondary">
     Skip to secondary menu
    </a>
   </li>
   <li>
    <a class="screen-reader-shortcut" href="#genesis-sidebar-primary">
     Saltar a la barra lateral principal
    </a>
   </li>
  </ul>
  <nav aria-label="Principal" class="nav-primary" id="genesis-nav-primary" ite

In [12]:
restaurante_text_data = []

for tag in soup.find_all(['h1', 'h2', 'h3', 'p']):
    if tag.name == "h1" and 'restaurantes' in tag.get_text(" ", strip=True).lower():
        restaurante_text_data.append(
            f"Titulo de la Página: {tag.get_text(' ', strip=True)}"
        )

    elif tag.name in ['h2', 'h3'] or (
        tag.name == 'p' and 'destaque' in tag.get('class', [])
    ):
        restaurante_text_data.append(
            f"Nombre/Destaque: {tag.get_text(' ', strip=True)}"
        )

    elif tag.name == 'p':

        if len(tag.get_text(" ", strip=True)) > 50:
            restaurante_text_data.append(
                f"Contenido: {tag.get_text(' ', strip=True)}"
            )

final_restaurante_data = "\n".join(restaurante_text_data)
final_restaurante_data = re.sub(r'\s+', ' ', final_restaurante_data).strip()

print(f"Website: {url}\n\n")
print(final_restaurante_data)

Website: https://www.conociendoitalia.com/8-tipos-de-locales-donde-comer-en-italia/


Contenido: Guía práctica y completa a los tipos de locales donde comer en Italia , definición y características de cada uno. Contenido: Si quieres comer algo en Italia, te encontrarás delante de una complicada elección pues los locales para comer tienen diferentes nombres: algunos que simplemente llegan de la tradición , otros que llaman la atención sobre lo que vas a comer o beber en aquel sitio particular. Contenido: Si quieres saber qué es una osteria , y cuál es la diferencia con una trattoria , ¡sígue leyendo!. Contenido: Veamos los 9 tipos de locales dónde comer en Italia, con su significado en español, que tipo de menú ofrecen y precios. Nombre/Destaque: 1. «OSTERIA» Contenido: La “Osteria” italiana es un ejercicio público, donde se sirven sobre todo vinos de diferentes tipos, y snacks en algunos casos; podemos decir que es como una taberna. Contenido: En la antiguedad ofrecia habitaciones a ba

In [13]:
import os
import re
from tavily import TavilyClient

api_key = os.environ.get("TAVILY_API_KEY")
if not api_key:
    raise ValueError("la clave TAVILY_API_KEY no fue encontrada.")

cliente_tavily = TavilyClient(api_key=api_key)

ciudad = "Tulum"
tavily_query = f"restaurantes en {ciudad} tripadvisor con mayor cantidad de reviews y rango de precios"

print("Iniciando la búsqueda agéntica por URLs de Tripadvisor con Tavily...")
tripadvisor_url = None

try:
    tavily_results = cliente_tavily.search(query=tavily_query, max_results=5)

    if tavily_results and tavily_results["results"]:
        print(f"Tavily encontró {len(tavily_results['results'])} resultados. Analizando...")
        for result in tavily_results["results"]:
            url = result["url"]
            if "tripadvisor.com" in url or "tripadvisor.com.mx" in url:
                tripadvisor_url = url
                break
    if not tripadvisor_url:
        print("Ninguna URL relevante de Tripadvisor fue encontrada en los primeros resultados.")
    else:
        print("Tavily no encontró resultados para la búsqueda agéntica.")
except Exception as e:
    print(f"Error en la búsqueda agéntica con Tavily: {e}. Verifica tu clave de API o tu conexión.")

if tripadvisor_url:
    clean_url = re.sub(r'-oa\d+-', '-', tripadvisor_url)
    tripadvisor_url = clean_url
    print(f"✅ URL encontrada limpia de paginación.")

print("*" * 50)
print(f"URL Final de Tripadvisor para el raspado: {tripadvisor_url if tripadvisor_url else 'NO ENCONTRADO'}")
print("*" * 50)

Iniciando la búsqueda agéntica por URLs de Tripadvisor con Tavily...
Tavily encontró 5 resultados. Analizando...
Tavily no encontró resultados para la búsqueda agéntica.
✅ URL encontrada limpia de paginación.
**************************************************
URL Final de Tripadvisor para el raspado: https://www.tripadvisor.com/Restaurant_Review-g23240074-d11932965-Reviews-Wild-Tulum_Beach_Tulum_Yucatan_Peninsula.html
**************************************************
